In [ ]:
from datasets import load_dataset

ds = load_dataset("indonesian-nlp/wikipedia-id-20231101")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/517 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/193M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/193M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/63.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/599059 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/66563 [00:00<?, ? examples/s]

In [ ]:
if isinstance(ds, dict) and 'train' in ds:
    idwiki_ds = ds['train']
else:
    idwiki_ds = ds

print(f"Dataset features: {idwiki_ds.features}")
print("\nFirst record example:")
print(idwiki_ds[0])

Dataset features: {'id': Value('string'), 'url': Value('string'), 'title': Value('string'), 'text': Value('string')}

First record example:
{'id': '3019350', 'url': 'https://id.wikipedia.org/wiki/Puteri%20Indonesia%20Sulawesi%20Tenggara', 'title': 'Puteri Indonesia Sulawesi Tenggara', 'text': 'Puteri Indonesia Sulawesi Tenggara adalah Kontes Kecantikan Nasional yang diadakan setiap tahun di Indonesia. Kontes ini memilih perwakilan, untuk bersaing dalam ajang Puteri Indonesia, yang merupakan salah satu kontes kecantikan terbesar di negara tersebut. Pemegang gelar Puteri Indonesia Sulawesi Tenggara saat ini adalah Sri Rahmisari Rembulan  dari Kota Kendari.\n\nSulawesi tenggara merupakan salah satu provinsi yang mengirimkan wakil setiap tahunnya dalam kontes Puteri Indonesia terhitung sejak debut pada tahun 1992. \n\nSulawesi Tenggara Telah Mendapatkan Penghargaan Dari Puteri Indonesia Yaitu Puteri Indonesia Favorit Pada 2006, Puteri Indonesia Intelegensia 3 & Puteri Indonesia Favorit Kep

In [ ]:
import pandas as pd
import re

In [ ]:
df_pandas = idwiki_ds.to_pandas()

WORLD_HISTORY_WHITELIST = [

    r"romawi", r"yunani kuno", r"mesir kuno", r"mesopotamia", r"babilonia",
    r"asiria", r"persia kuno", r"fenisia", r"kartago", r"sumeria",
    r"akkadia", r"hurria", r"hittit", r"minoam", r"miken",
    r"nubia", r"aksumit", r"punt",

    # Asian empires & dynasties
    r"dinasti (han|tang|song|ming|qing|yuan|sui|zhou|qin|shang|jin|liao|xia)",
    r"kekaisaran (tiongkok|china|cina|mongol|jepang|korea)",
    r"keshogunan", r"samurai", r"shogun",
    r"dinasti (joseon|goryeo|silla|baekje|goguryeo)",
    r"dinasti (maurya|gupta|mughal|maratha|chola|pallava|chalukya)",

    # Middle East & Islamic empires
    r"kekhalifahan (abbasiyah|umayyah|fatimiyah|rasyidin)",
    r"ottoman", r"turki usmani", r"sassanid", r"safawi", r"timurid",

    # European history
    r"kekaisaran romawi", r"kekaisaran bizantium", r"byzantium", r"bizantium",
    r"kekaisaran frank", r"kekaisaran suci romawi", r"kekaisaran karoling",
    r"kerajaan (prancis|inggris|spanyol|portugal|austria|prusia|swedia|polandia|rusia)",
    r"revolusi prancis", r"revolusi industri", r"revolusi (amerika|rusia|cina|tiongkok)",
    r"napoleon", r"perang salib", r"abad pertengahan", r"renaisans", r"reformasi protestan",
    r"zaman pencerahan", r"absolutisme", r"feodalisme eropa",

    # The Americas
    r"(maya|aztec|inca|olmec|toltec|zapotec|mississippian)",
    r"peradaban (maya|aztec|inca|prekolumbia|mesoamerika|andean)",
    r"penaklukan spanyol", r"conquistador",

    # Africa
    r"(mali|songhai|ghana kuno|kerajaan kongo|zimbabwe kuno|carthage)",
    r"peradaban afrika",

    # World Wars & Modern World History
    r"perang dunia (i|ii|1|2|pertama|kedua)",
    r"perang dingin",
    r"holocaust", r"nazi", r"fasisme", r"komunisme soviet",
    r"uni soviet", r"perang korea", r"perang vietnam",
    r"perang (napoleon|saudara amerika|kemerdekaan amerika)",

    # Prehistoric & cross-civilizational
    r"zaman (batu|perunggu|besi|paleolitik|neolitik|mesolitik)",
    r"manusia purba", r"homo sapiens", r"neanderthal",
    r"peradaban kuno", r"sejarah dunia", r"sejarah kuno",
    r"kolonialisme eropa", r"imperialisme eropa", r"penjelajahan eropa",
    r"jalur sutra", r"perdagangan dunia kuno",
]

HARD_EXCLUDE_TITLE = [
    # Indonesian geography & admin
    r"\b(indonesia|jawa|sumatra|sumatera|kalimantan|sulawesi|papua|bali|lombok|maluku|nusa tenggara|aceh|riau|jambi|bengkulu|lampung)\b",
    r"^(kota|kabupaten|provinsi|desa|kecamatan|kelurahan|distrik)\b",
    r"^(sungai|gunung|pulau|danau|selat|teluk|tanjung|cape|laut)\b",
    r"^(jalan|stasiun|bandara|pelabuhan|terminal)\b",

    # Indonesian kingdoms — explicit list
    r"(sriwijaya|majapahit|mataram|demak|pajang|banten|cirebon|aceh|ternate|tidore|gowa|bone|buton|kutai|tarumanagara|pajajaran|singhasari|kediri|jenggala|kahuripan|kanjuruhan|medang|singosari|tumapel|kalingga|holing|salakanagara|kendan|galuh|sunda|indrapura|pagaruyung|minangkabau|dharmasraya|melayu kuno|langkasuka)",

    # Malay / regional Southeast Asian (non-world-history)
    r"(melayu|bugis|makassar|dayak|batak|sunda|minang|toraja|sasak|baduy|asmat|dani|ambon|ternate|tidore)",

    # Indonesian national history
    r"(kemerdekaan indonesia|proklamasi|soekarno|soeharto|orde baru|orde lama|reformasi indonesia|pergerakan nasional indonesia|budi utomo|sumpah pemuda|pki|g30s)",

    # People / biography
    r"(tokoh|biografi|politikus|aktivis|artis|aktor|aktris|penyanyi|musisi|atlet|olahragawan)",

    # Entertainment & sports (more comprehensive)
    r"(film|movie|bioskop|sutradara|aktor|aktris|pemain|karakter|episode|musim|season|soundtrack|ost|lagu|album|sinetron|serial|acara tv|klub|tim|sepak bola|basket|bulu tangkis|liga|turnamen|novel|komik|game|permainan)",

    # Modern institutions, companies
    r"(perusahaan|pt\.|tbk\.|grup|holding|bank|universitas|sekolah|rumah sakit)",
]

In [ ]:
def matches_any(text: str, patterns: list, flags=re.IGNORECASE) -> bool:
    if not isinstance(text, str):
        return False
    return any(re.search(p, text, flags) for p in patterns)

def whitelist_density(text: str, min_hits: int = 3) -> bool:
    """Require at least N distinct whitelist pattern matches in the body."""
    if not isinstance(text, str):
        return False
    return sum(1 for p in WORLD_HISTORY_WHITELIST if re.search(p, text, re.IGNORECASE)) >= min_hits

def is_world_history(row, min_text_length: int = 200) -> bool:
    title = str(row.get("title", "") or "")
    text  = str(row.get("text",  "") or "")

    if len(text.strip()) < min_text_length:
        return False

    if matches_any(title, HARD_EXCLUDE_TITLE):
        return False

    title_matches = matches_any(title, WORLD_HISTORY_WHITELIST)
    if not title_matches:
        return False

    return whitelist_density(text, min_hits=2)

In [ ]:
mask = df_pandas.apply(lambda row: is_world_history(row, min_text_length=550), axis=1)
world_history_df = df_pandas[mask].reset_index(drop=True)

print(f"Total articles in dataset  : {len(df_pandas):,}")
print(f"World history articles found: {len(world_history_df):,}")
print("\nSample titles:")
for t in world_history_df["title"].head(30).tolist():
    print(" •", t)

Total articles in dataset  : 599,059
World history articles found: 694

Sample titles:
 • Die Tochter des Samurai
 • Bahasa Arab Mesopotamia Utara
 • Daftar Pahlawan Uni Soviet (B)
 • Hubungan Romawi dengan Tiongkok
 • Bahasa Akkadia
 • Republik Romawi
 • Hidangan abad pertengahan
 • Angkatan Laut Kekaisaran Tiongkok
 • Sejarah Uni Soviet (1953–1964)
 • Friedrich I, Kaisar Romawi Suci
 • Feminazi
 • Rumania pada Perang Dunia II
 • Wanita di Jerman Nazi
 • Ramayana (seri televisi 2012)
 • Ptolemais di Fenisia
 • Region di Somalia
 • Perang Salib Kedua
 • Bahasa Yunani Mikenai
 • Kerajaan Napoli (Napoleon)
 • Uni Soviet pada Perang Korea
 • Keruntuhan Zaman Perunggu Akhir
 • Perang Salib
 • Penindasan orang Serbia selama Perang Dunia II
 • 108 Martir Perang Dunia II
 • Daftar Kaisar Romawi
 • Karl VI, Kaisar Romawi Suci
 • Mycomya tolteca
 • Naram-Sin dari Akkadia
 • Kekristenan pada Abad Pertengahan
 • Sejarah Kartago


In [ ]:
display(world_history_df)

,id,url,title,text
0,3941550,https://id.wikipedia.org/wiki/Die%20Tochter%20...,Die Tochter des Samurai,"The Daughter of the Samurai (, bahasa Jepang: ..."
1,3098295,https://id.wikipedia.org/wiki/Bahasa%20Arab%20...,Bahasa Arab Mesopotamia Utara,Bahasa Arab Mesopotamia Utara (juga dikenali s...
2,2783878,https://id.wikipedia.org/wiki/Daftar%20Pahlawa...,Daftar Pahlawan Uni Soviet (B),Berikut ini adalah daftar Pahlawan Uni Soviet:...
3,2312618,https://id.wikipedia.org/wiki/Hubungan%20Romaw...,Hubungan Romawi dengan Tiongkok,Hubungan Romawi dengan Tiongkok mengacu kepada...
4,144942,https://id.wikipedia.org/wiki/Bahasa%20Akkadia,Bahasa Akkadia,Bahasa Akkadia (lišānum akkadītum) ak-ka-du-u...
...,...,...,...,...
689,2628331,https://id.wikipedia.org/wiki/Agama%20Mesir%20...,Agama Mesir Kuno,Agama Mesir Kuno adalah bentuk kepercayaan dan...
690,434439,https://id.wikipedia.org/wiki/Senat%20Republik...,Senat Republik Romawi,Senat Republik Romawi merupakan institusi poli...
691,2312213,https://id.wikipedia.org/wiki/Oxford%20Diction...,Oxford Dictionary of Byzantium,The Oxford Dictionary of Byzantium (sering dis...
692,2426959,https://id.wikipedia.org/wiki/Hubungan%20Somal...,Hubungan Somalia dengan Tiongkok,"Hubungan Tiongkok–Somalia (, ) merujuk kepada ..."


In [ ]:
import regex as re
import numpy as np

print("--- Noise Analysis Report (Before Cleaning) ---")
print(f"Total rows: {len(world_history_df)}\n")

def count_pattern(df, column, pattern):
    return df[column].astype(str).apply(lambda x: bool(re.search(pattern, x))).sum()

multi_space_count = count_pattern(world_history_df, 'text', r'  +')
print(f"- Rows with multiple consecutive spaces in 'text': {multi_space_count:,}")

non_latin_count = count_pattern(world_history_df, 'text', r'[^\p{Latin}\p{P}\p{S}\p{Z}\p{N}\p{C}]')
print(f"- Rows with non-Latin script characters in 'text': {non_latin_count:,}")

special_symbols_count = count_pattern(world_history_df, 'text', r'[^\w\s.,!?-]')
print(f"- Rows with unusual special symbols in 'text': {special_symbols_count:,}")

mediawiki_count = count_pattern(world_history_df, 'text', r'(\{\{|\}\}|\[\[|\]\]|==[^=]+==)')
print(f"- Rows with MediaWiki template syntax ({{, }}, [[, ]], ==header==) in 'text': {mediawiki_count:,}")

html_count = count_pattern(world_history_df, 'text', r'(<[^>]+>|&[a-zA-Z0-9#]+;)')
print(f"- Rows with HTML tags or entities in 'text': {html_count:,}")

text_lengths = world_history_df['text'].astype(str).apply(len)
print("\n--- Text Length Statistics (Before Cleaning) ---")
print(f"- Min text length: {np.min(text_lengths):,}")
print(f"- Max text length: {np.max(text_lengths):,}")
print(f"- Mean text length: {np.mean(text_lengths):,.2f}")
print(f"- Median text length: {np.median(text_lengths):,.2f}")

--- Noise Analysis Report (Before Cleaning) ---
Total rows: 694

- Rows with multiple consecutive spaces in 'text': 675
- Rows with non-Latin script characters in 'text': 84
- Rows with unusual special symbols in 'text': 674
- Rows with MediaWiki template syntax ({, }, [[, ]], ==header==) in 'text': 19
- Rows with HTML tags or entities in 'text': 22

--- Text Length Statistics (Before Cleaning) ---
- Min text length: 548
- Max text length: 142,492
- Mean text length: 8,435.28
- Median text length: 2,730.00


In [ ]:
import html

world_history_clean_df = world_history_df.copy()

def strip_mediawiki_templates(text):
    # Remove {{...}} blocks (multiline)
    text = re.sub(r'\{\{.*?\}\}', '', text, flags=re.DOTALL)
    # Remove [[File:...]] blocks
    text = re.sub(r'\[\[File:.*?\].*?\]\]', '', text) # Also remove File:...|alt text
    # Unwrap [[...|display]] to keep display text only
    text = re.sub(r'\[\[[^|\]]+\|(.*?)\]\]', r'\1', text)
    # Remove [[...]] links, keeping only the text inside
    text = re.sub(r'\[\[(.*?)\]\]', r'\1', text)
    # Remove ==...== section headers
    text = re.sub(r'==.*?==', '', text)
    return text

def remove_html_tags_and_entities(text):
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Decode HTML entities
    text = html.unescape(text)
    return text

def remove_non_latin_scripts(text):
    cleaned_text = re.sub(r'[^\p{L}\p{N}\p{P}\p{Z}\p{M}\s.,!?;:\-\'"()/\u0024\u00A2-\u00A5\u20A0-\u20BD]', '', text)
    return cleaned_text

def collapse_and_strip_whitespace(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

print("Cleaning 'text' column...")
world_history_clean_df['text'] = world_history_clean_df['text'].astype(str)
world_history_clean_df['text'] = world_history_clean_df['text'].apply(strip_mediawiki_templates)
world_history_clean_df['text'] = world_history_clean_df['text'].apply(remove_html_tags_and_entities)
world_history_clean_df['text'] = world_history_clean_df['text'].apply(remove_non_latin_scripts)
world_history_clean_df['text'] = world_history_clean_df['text'].apply(collapse_and_strip_whitespace)

print("Cleaning 'title' column...")
world_history_clean_df['title'] = world_history_clean_df['title'].astype(str)
world_history_clean_df['title'] = world_history_clean_df['title'].apply(collapse_and_strip_whitespace)

initial_row_count = len(world_history_clean_df)

print("Applying filters and dropping rows...")
world_history_clean_df = world_history_clean_df[world_history_clean_df['text'].apply(len) >= 300]

exact_titles_to_drop = [
    "Piala Maya 2012", "Piala Maya 2013", "Piala Maya 2021",
    "Mycomya tolteca", "Eurysternus inca", "Diamesa incallidus",
    "Onthophagus zapotecus", "Psilocybe aztecorum", "Ibex nubia",
    "Bombylius aztec", "Acronyches maya", "Mallophora inca",
    "Onthophagus incantatus", "Phthiria olmeca", "Phthiria aztec",
    "Ctenodontina maya", "Puntius", "Dichotomius maya",
    "Phanaeus zapotecus", "Bombylius incanus", "Hadroneura kincaidi",
    "Agromyza kincaidi", "Thyridanthrax incanus", "Leucothrix incana",
    "Anastrepha maya", "Mallophora incanipes", "Apolephthisa subincana",
    "Eurysternus maya", "Long Kemuat, Bahau Hulu, Malinau",
    "Mayang, Gatak, Sukoharjo", "Somalia dalam tahun 2014",
    "Sool, Somalia", "Puntland", "Somaliland", "Goga Ashkenazi",
    "Moazzam Malik", "Nazif Basir", "Mohammad Nazir", "Adam Malik",
    "Kiki Amalia", "Vladimir Mayakovsky", "Oba: The Last Samurai",
    "The Last Samurai", "Seven Samurai", "Ramayana (seri televisi 2012)",
    "Samurai Warriors 4", "Shogun Assassin", "Maya Island Air",
    "Compagnia Nazionale Aeronautica", "Ramayana Lestari Sentosa",
    "Divisione Nazionale", "Divisione Nazionale 1928–1929",
    "Divisione Nazionale 1945–1946", "Autodromo Nazionale di Monza",
    "Stadion Nazionale", "Abnormalitas", "Mamalia", "Opuntia",
    "Ujaran kebencian di dunia maya"
]
world_history_clean_df = world_history_clean_df[~world_history_clean_df['title'].isin(exact_titles_to_drop)]

keywords_to_check = ["perang", "dinasti", "kerajaan", "kekaisaran", "revolusi", "kolonial",
                       "kemerdekaan", "pertempuran", "abad", "raja", "sultan", "perjanjian",
                       "sejarah", "war", "battle", "dynasty", "empire", "history",
                       "kingdom", "revolution", "ancient", "medieval", "conquest", "treaty", "civilization"]

keyword_pattern = r'|'.join([re.escape(k) for k in keywords_to_check])

world_history_clean_df = world_history_clean_df[
    world_history_clean_df.apply(lambda row: bool(re.search(keyword_pattern, row['title'], re.IGNORECASE)) or
                                              bool(re.search(keyword_pattern, row['text'], re.IGNORECASE)), axis=1)
]


print("\n--- Cleaning Report ---")
print(f"Initial rows: {initial_row_count:,}")
print(f"Final rows: {len(world_history_clean_df):,}")

mean_text_length_before = world_history_df['text'].astype(str).apply(len).mean()
mean_text_length_after = world_history_clean_df['text'].astype(str).apply(len).mean()
print(f"Mean text length before cleaning: {mean_text_length_before:,.2f}")
print(f"Mean text length after cleaning: {mean_text_length_after:,.2f}")

remaining_mediawiki_percent = (world_history_clean_df['text'].str.contains(r'\{\{') | world_history_clean_df['text'].str.contains(r'\}\}')).mean() * 100
print(f"% of rows still containing '{{' or '}}' after cleaning: {remaining_mediawiki_percent:.2f}%")

remaining_multi_space_percent = world_history_clean_df['text'].str.contains(r'  +').mean() * 100
print(f"% of rows still containing multiple spaces after cleaning: {remaining_multi_space_percent:.2f}%")

print("\n--- Sample Cleaned Rows ---")
for i in range(min(3, len(world_history_clean_df))):
    row = world_history_clean_df.iloc[i]
    print(f"Title: {row['title']}")
    print(f"Text (first 300 chars): {row['text'][:300]}...")
    print("---------------------------------------------------")

cleaned_csv_filename = 'world_history_articles_cleaned.csv'
world_history_clean_df.to_csv(cleaned_csv_filename, index=False)
print(f"\nCleaned DataFrame saved to '{cleaned_csv_filename}' with {len(world_history_clean_df):,} rows.")

Cleaning 'text' column...
Cleaning 'title' column...
Applying filters and dropping rows...

--- Cleaning Report ---
Initial rows: 694
Final rows: 601
Mean text length before cleaning: 8,435.28
Mean text length after cleaning: 8,869.18
% of rows still containing '{' or '}' after cleaning: 0.33%
% of rows still containing multiple spaces after cleaning: 0.00%

--- Sample Cleaned Rows ---
Title: Die Tochter des Samurai
Text (first 300 chars): The Daughter of the Samurai (, bahasa Jepang: ) adalah film drama Jerman-Jepang tahun 1937 yang disutradarai oleh Arnold Fanck dan Mansaku Itami dan dibintangi oleh Setsuko Hara, Ruth Eweler dan Sessue Hayakawa. Film ini adalah film pertama dari dua film produksi bersama antara Kekaisaran Jepang dan...
---------------------------------------------------
Title: Bahasa Arab Mesopotamia Utara
Text (first 300 chars): Bahasa Arab Mesopotamia Utara (juga dikenali sebagai Moslawi [artinya 'Mosul'] atau Bahasa Arab Qeltu Mesopotamia) ialah varietas bahasa Ar

In [ ]:
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)
# display(world_history_clean_df)

In [ ]:
unique_titles = world_history_clean_df['title'].unique()
print(f"Total unique titles: {len(unique_titles)}")
for title in unique_titles:
    print(f"- {title}")

Total unique titles: 601
- Die Tochter des Samurai
- Bahasa Arab Mesopotamia Utara
- Daftar Pahlawan Uni Soviet (B)
- Hubungan Romawi dengan Tiongkok
- Bahasa Akkadia
- Republik Romawi
- Hidangan abad pertengahan
- Angkatan Laut Kekaisaran Tiongkok
- Sejarah Uni Soviet (1953–1964)
- Friedrich I, Kaisar Romawi Suci
- Rumania pada Perang Dunia II
- Wanita di Jerman Nazi
- Ptolemais di Fenisia
- Perang Salib Kedua
- Bahasa Yunani Mikenai
- Kerajaan Napoli (Napoleon)
- Uni Soviet pada Perang Korea
- Keruntuhan Zaman Perunggu Akhir
- Perang Salib
- Penindasan orang Serbia selama Perang Dunia II
- 108 Martir Perang Dunia II
- Daftar Kaisar Romawi
- Karl VI, Kaisar Romawi Suci
- Naram-Sin dari Akkadia
- Kekristenan pada Abad Pertengahan
- Sejarah Kartago
- Alfabet Yunani Kuno
- Pemboikotan Nazi terhadap bisnis Yahudi
- Perang Dunia I
- Reformasi Protestan di Italia
- Rumpun bahasa Maya
- Pemberontakan Katolik terhadap Jerman Nazi
- Korea di bawah Pemerintahan Kekaisaran Jepang
- Historiografi

In [ ]:
import html

world_history_final_df = world_history_clean_df.copy()

original_text_for_comparison = world_history_final_df['text'].copy()

def apply_remaining_mediawiki_fixes(text):
    old_text = ''
    while old_text != text:
        old_text = text
        text = re.sub(r'\{\{[^{}]*\}\}', '', text, flags=re.DOTALL)
    old_text = ''
    while old_text != text:
        old_text = text
        text = re.sub(r'\{\{.*?\}\}', '', text, flags=re.DOTALL)

    old_text = ''
    while old_text != text:
        old_text = text
        text = re.sub(r'\[\[[^|\]]+\|(.*?)\]\]', r'\1', text)

    old_text = ''
    while old_text != text:
        old_text = text
        text = re.sub(r'\[\[(.*?)\]\]', r'\1', text)


def clean_empty_parentheses_artifacts(text):
    text = re.sub(r'\(\s*[,;:\s]*\s*\)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Applying additional fixes to 'text' column...")
world_history_final_df['text'] = world_history_final_df['text'].apply(apply_remaining_mediawiki_fixes)
world_history_final_df['text'] = world_history_final_df['text'].apply(clean_empty_parentheses_artifacts)

world_history_final_df['text'] = world_history_final_df['text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

print("\n--- Verification Report (After Additional Fixes) ---")

print(f"Row count before these fixes: {len(world_history_clean_df):,}")
print(f"Row count after these fixes: {len(world_history_final_df):,} (should be the same)")

remaining_double_brace_after_fix = world_history_final_df['text'].str.contains(r'\{\{').sum()
print(f"Rows still containing '{{' after fix: {remaining_double_brace_after_fix}")

remaining_double_bracket_after_fix = world_history_final_df['text'].str.contains(r'\[\[').sum()
print(f"Rows still containing '[[' after fix: {remaining_double_bracket_after_fix}")

remaining_empty_parens_after_fix = world_history_final_df['text'].str.contains(r'\(\s*[,;:\s]*\s*\)').sum()
print(f"Rows still containing empty parentheses after fix: {remaining_empty_parens_after_fix}")

affected_indices = original_text_for_comparison.index[original_text_for_comparison != world_history_final_df['text']]

print(f"\n{len(affected_indices)} rows had their 'text' content modified by these fixes.")
print("\n--- Sample Affected Rows (Title + First 400 chars of Fixed Text) ---")

sample_rows_to_display = world_history_final_df.loc[affected_indices].head(3)

if not sample_rows_to_display.empty:
    for index, row in sample_rows_to_display.iterrows():
        print(f"Title: {row['title']}")
        print(f"Text (first 400 chars): {row['text'][:400]}...")
        print("---------------------------------------------------")
else:
    print("No text content was modified by these fixes, or no sample could be drawn.")

final_csv_filename = 'world_history_articles_final.csv'
world_history_final_df.to_csv(final_csv_filename, index=False)
print(f"\nFinal DataFrame saved to '{final_csv_filename}' with {len(world_history_final_df):,} rows.")

Applying additional fixes to 'text' column...

--- Verification Report (After Additional Fixes) ---
Row count before these fixes: 601
Row count after these fixes: 601 (should be the same)
Rows still containing '{' after fix: 1
Rows still containing '[[' after fix: 2
Rows still containing empty parentheses after fix: 0

114 rows had their 'text' content modified by these fixes.

--- Sample Affected Rows (Title + First 400 chars of Fixed Text) ---
Title: Hubungan Romawi dengan Tiongkok
Text (first 400 chars): Hubungan Romawi dengan Tiongkok mengacu kepada kontak yang utamanya tak langsung antara Kekaisaran Romawi dan Dinasti Han dan kemudian antara Kekaisaran Romawi Timur dengan berbagai dinasti dalam sejarah Tiongkok. Hubungan yang berlangsung dapat berupa pertukaran barang dagang serta informasi dan terkadang juga mencakup kedatangan pengelana dan pengiriman utusan. Dalam sejarahnya, Romawi terus mem...
---------------------------------------------------
Title: Republik Romawi
Text (fi

In [ ]:
world_history_v2_df = world_history_final_df.copy()

def strip_wiki_templates(text):
    prev = None
    while prev != text:
        prev = text
        text = re.sub(r'\{\{[^\{\}]*\}\}', '', text, flags=re.DOTALL)

    text = re.sub(r'\{\{|\}\}', '', text)
    return text

world_history_v2_df['text'] = world_history_v2_df['text'].apply(strip_wiki_templates)

world_history_v2_df['text'] = world_history_v2_df['text'].str.replace(r'\[\[|\]\]', '', regex=True)

world_history_v2_df['text'] = world_history_v2_df['text'].str.replace(
    r'\(\s*[,;:\s]+\s*\)', '', regex=True)

world_history_v2_df['text'] = world_history_v2_df['text'].str.replace(
    r',\s*\)', ')', regex=True)
world_history_v2_df['text'] = world_history_v2_df['text'].str.replace(
    r';\s*\)', ')', regex=True)

world_history_v2_df['text'] = world_history_v2_df['text'].str.replace(
    r' {2,}', ' ', regex=True).str.strip()

world_history_v2_df['text'] = world_history_v2_df['text'].str.replace(
    '•', '\n', regex=False)

checks = {
    'Wiki {{ }} remaining':          world_history_v2_df['text'].str.contains(r'\{\{|\}\}', regex=True).sum(),
    'Wiki [[ ]] remaining':          world_history_v2_df['text'].str.contains(r'\[\[|\]\]', regex=True).sum(),
    'Trailing ,) remaining':         world_history_v2_df['text'].str.contains(r',\s*\)', regex=True).sum(),
    'Empty paren artifacts remaining': world_history_v2_df['text'].str.contains(r'\(\s*[,;:\s]*\)', regex=True).sum(),
    'Bullet • remaining':            world_history_v2_df['text'].str.contains('•', regex=False).sum(),
    'Multiple spaces remaining':     world_history_v2_df['text'].str.contains(r'  +', regex=True).sum(),
}

print("\n--- Verification Report ---")
for label, count in checks.items():
    status = '✓ CLEAN' if count == 0 else f'✗ {count} rows still affected'
    print(f'{label}: {status}')

print(f'\nRow count: {len(world_history_v2_df)} (expected: 601 — no rows should be dropped)')

# Save the final DataFrame
world_history_v2_df.to_csv('world_history_articles_v2.csv', index=False)
print('Saved: world_history_articles_v2.csv')


--- Verification Report ---
Wiki {{ }} remaining: ✓ CLEAN
Wiki [[ ]] remaining: ✓ CLEAN
Trailing ,) remaining: ✓ CLEAN
Empty paren artifacts remaining: ✓ CLEAN
Bullet • remaining: ✓ CLEAN
Multiple spaces remaining: ✓ CLEAN

Row count: 601 (expected: 601 — no rows should be dropped)
Saved: world_history_articles_v2.csv


In [ ]:
import pandas as pd

world_history_v2_df = pd.read_csv('world_history_articles_v2.csv')
display(world_history_v2_df.head())

id  \
0  3941550   
1  3098295   
2  2783878   
3  2312618   
4   144942   

                                                                        url  \
0               https://id.wikipedia.org/wiki/Die%20Tochter%20des%20Samurai   
1         https://id.wikipedia.org/wiki/Bahasa%20Arab%20Mesopotamia%20Utara   
2  https://id.wikipedia.org/wiki/Daftar%20Pahlawan%20Uni%20Soviet%20%28B%29   
3       https://id.wikipedia.org/wiki/Hubungan%20Romawi%20dengan%20Tiongkok   
4                            https://id.wikipedia.org/wiki/Bahasa%20Akkadia   

                             title  \
0          Die Tochter des Samurai   
1    Bahasa Arab Mesopotamia Utara   
2   Daftar Pahlawan Uni Soviet (B)   
3  Hubungan Romawi dengan Tiongkok   
4                   Bahasa Akkadia   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [ ]:
text_lengths_v2 = world_history_v2_df['text'].astype(str).apply(len)

print("--- Text Length Statistics (world_history_v2_df) ---")
print(f"- Min text length: {text_lengths_v2.min():,}")
print(f"- Max text length: {text_lengths_v2.max():,}")
print(f"- Mean text length: {text_lengths_v2.mean():,.2f}")
print(f"- Median text length: {text_lengths_v2.median():,.2f}")

--- Text Length Statistics (world_history_v2_df) ---
- Min text length: 528
- Max text length: 142,285
- Mean text length: 8,868.00
- Median text length: 2,935.00


In [ ]:
text_lengths_v2 = world_history_v2_df['text'].astype(str).apply(len)

print("--- Text Length Statistics (Content) ---")
print(f"- Panjang teks minimum: {text_lengths_v2.min():,}")
print(f"- Panjang teks maksimum: {text_lengths_v2.max():,}")
print(f"- Panjang teks rata-rata: {text_lengths_v2.mean():,.2f}")
print(f"- Panjang teks median: {text_lengths_v2.median():,.2f}")

--- Text Length Statistics (Content) ---
- Panjang teks minimum: 528
- Panjang teks maksimum: 142,285
- Panjang teks rata-rata: 8,868.00
- Panjang teks median: 2,935.00


In [ ]:
cleaned_historical_summarization_df = world_history_v2_df.copy()

FINAL_EXCLUDE_TITLE_LIST = [
    "Daftar Pahlawan Uni Soviet (B)", "Daftar Kaisar Romawi", "Daftar Pahlawan Uni Soviet (S)",
    "Daftar samurai pada Zaman Sengoku", "Daftar Kekuatan Besar abad pertengahan", "Daftar Raja Sumeria",
    "Daftar Skuadron Udara Angkatan Darat Kekaisaran Jepang", "Daftar konferensi Sekutu Perang Dunia II",
    "Daftar kamp konsentrasi Nazi", "Daftar penulis yang dilarang di Jerman Nazi",
    "Daftar penerbang ulung dalam Perang Dunia II dari Jepang", "Daftar pesawat militer Jerman dalam Perang Dunia II",
    "Daftar ahli geografi Yunani-Romawi", "Daftar kapal perang Angkatan Laut Kekaisaran Jepang",
    "Daftar peradaban kuno", "Daftar pertempuran pada Perang Dunia I",
    "Daftar pesawat jet pada Perang Dunia II", "Daftar negara yang terlibat dalam Perang Dunia II",
    "Daftar senjata infanteri pada Perang Dunia II", "Daftar operasi militer Perang Dunia II",
    "Daftar dewa-dewi Romawi", "Daftar pertemuan puncak Konferensi Tingkat Tinggi Uni Soviet–Amerika Serikat",
    "Daftar pemimpin Uni Soviet", "Daftar perlengkapan peperangan elektronika dalam Perang Dunia II",
    "Daftar peluru kendali Jerman dalam Perang Dunia II", "Daftar permaisuri Romawi dan Bizantium",
    "Urutan waktu peristiwa sebelum Perang Dunia II", "Urutan waktu Perang Dunia II (1940)",
    "Urutan waktu Perang Dunia II (1942)", "Urutan waktu Perang Dunia II (1943)",
    "Urutan waktu Perang Dunia II (1944)", "Urutan waktu Perang Dunia II (1945)",
    "Garis waktu Perang Dunia II (1941)", "Peristiwa sebelum Perang Dunia II di Eropa",
    "Onthophagus maya", "Phthiria toltec", "Exoprosopa aztec", "Lochmorhynchus puntarenensis",
    "Lecania inca", "Diamesa incallida", "Onthophagus punthari", "Anjing gembala Belgia (Malinois)",
    "Maya Rudolph", "Mayavi Maling", "Suzuki Shogun", "Maha Guru Manikmaya", "Nishi Muku Samurai",
    "Nazir", "Kerajaan Polandia (disambiguasi)", "Maria Amalia",
    "Pondok Pesantren Subulussalam, Sayurmaincat", "Kemah, Erzincan"
]

initial_row_count_before_filtering = len(cleaned_historical_summarization_df)
print(f"Jumlah artikel sebelum filtering judul: {initial_row_count_before_filtering:,}")

cleaned_historical_summarization_df = cleaned_historical_summarization_df[~cleaned_historical_summarization_df['title'].isin(FINAL_EXCLUDE_TITLE_LIST)]

cleaned_historical_summarization_df['text_length'] = cleaned_historical_summarization_df['text'].astype(str).apply(len)

cleaned_historical_summarization_df = cleaned_historical_summarization_df[
    (cleaned_historical_summarization_df['text_length'] >= 1000) &
    (cleaned_historical_summarization_df['text_length'] <= 20000)
]

cleaned_historical_summarization_df['text'] = cleaned_historical_summarization_df['text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

final_row_count = len(cleaned_historical_summarization_df)
removed_row_count = initial_row_count_before_filtering - final_row_count

print("\n--- Laporan Analisis Dataset ---")
print(f"Total data sebelum semua filtering: {initial_row_count_before_filtering:,}")
print(f"Total data setelah semua filtering: {final_row_count:,}")
print(f"Jumlah data terhapus: {removed_row_count:,}")

final_text_lengths = cleaned_historical_summarization_df['text_length']
print("\nStatistik Panjang Karakter (setelah filtering):")
print(f"  - Panjang teks minimum: {final_text_lengths.min():,}")
print(f"  - Panjang teks maksimum: {final_text_lengths.max():,}")
print(f"  - Panjang teks rata-rata: {final_text_lengths.mean():,.2f}")
print(f"  - Panjang teks median: {final_text_lengths.median():,.2f}")

final_output_filename = 'cleaned_historical_summarization_dataset.csv'
cleaned_historical_summarization_df.drop(columns=['text_length']).to_csv(final_output_filename, index=False)
print(f"\nDataset akhir disimpan ke '{final_output_filename}' dengan {final_row_count:,} baris.")

print("\n--- 5 Contoh Sampel Baris Hasil Akhir Dataset ---")
display(cleaned_historical_summarization_df.drop(columns=['text_length']).head(5))

Jumlah artikel sebelum filtering judul: 601

--- Laporan Analisis Dataset ---
Total data sebelum semua filtering: 601
Total data setelah semua filtering: 418
Jumlah data terhapus: 183

Statistik Panjang Karakter (setelah filtering):
  - Panjang teks minimum: 1,000
  - Panjang teks maksimum: 19,029
  - Panjang teks rata-rata: 4,600.23
  - Panjang teks median: 3,231.50

Dataset akhir disimpan ke 'cleaned_historical_summarization_dataset.csv' dengan 418 baris.

--- 5 Contoh Sampel Baris Hasil Akhir Dataset ---


id  \
0  3941550   
1  3098295   
4   144942   
5   215850   
8  2805805   

                                                                              url  \
0                     https://id.wikipedia.org/wiki/Die%20Tochter%20des%20Samurai   
1               https://id.wikipedia.org/wiki/Bahasa%20Arab%20Mesopotamia%20Utara   
4                                  https://id.wikipedia.org/wiki/Bahasa%20Akkadia   
5                                 https://id.wikipedia.org/wiki/Republik%20Romawi   
8  https://id.wikipedia.org/wiki/Sejarah%20Uni%20Soviet%20%281953%E2%80%931964%29   

                            title  \
0         Die Tochter des Samurai   
1   Bahasa Arab Mesopotamia Utara   
4                  Bahasa Akkadia   
5                 Republik Romawi   
8  Sejarah Uni Soviet (1953–1964)   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [ ]:
import pandas as pd
import re

cleaned_historical_summarization_df = pd.read_csv('cleaned_historical_summarization_dataset.csv')

BIBLIO_PATTERNS = [
    r'\b(Daftar pustaka|Referensi|Rujukan|Pranala luar|Lihat pula|Sumber|Catatan kaki)\b',
    r'==\s*(Daftar Pustaka|Referensi|Rujukan|Pranala Luar|Lihat Pula|Sumber|Catatan Kaki)\s*==',
    r'\b(Bibliografi|Bibliografis|Eksternal links|Further reading)\b' # Including some common English terms that might remain or appear
]

combined_biblio_pattern = '|'.join(BIBLIO_PATTERNS)

def contains_non_narrative_sections(text, patterns, min_section_matches=1):
    """Checks if the text contains a significant number of non-narrative section indicators."""
    matches = re.findall(patterns, text, re.IGNORECASE)
    # Check for actual section headers or prominent usage rather than just keywords
    return len(matches) >= min_section_matches

list_title_pattern = r'^Daftar\s+.*'
initial_row_count = len(cleaned_historical_summarization_df)

mask_to_keep = ~cleaned_historical_summarization_df.apply(
    lambda row: (
        re.search(list_title_pattern, row['title'], re.IGNORECASE) is not None or
        contains_non_narrative_sections(row['text'], combined_biblio_pattern)
    ),
    axis=1
)

filtered_df = cleaned_historical_summarization_df[mask_to_keep].reset_index(drop=True)

removed_count = initial_row_count - len(filtered_df)

print(f"Initial number of articles: {initial_row_count:,}")
print(f"Articles removed due to non-narrative/bibliography content: {removed_count:,}")
print(f"Final number of articles: {len(filtered_df):,}")

print("\n--- Sample of cleaned articles (first 5 rows) ---")
display(filtered_df.head())

final_output_filename = 'cleaned_historical_summarization_dataset_v2.csv'
filtered_df.to_csv(final_output_filename, index=False)
print(f"\nFinal cleaned dataset saved to '{final_output_filename}'.")

Initial number of articles: 418
Articles removed due to non-narrative/bibliography content: 402
Final number of articles: 16

--- Sample of cleaned articles (first 5 rows) ---


,id,url,title,text
0,2354766,https://id.wikipedia.org/wiki/Sejarah%20Dinast...,Sejarah Dinasti Ming,Dinasti Ming (23 Januari 1368 – 25 April 1644)...
1,22930,https://id.wikipedia.org/wiki/Dinasti%20Qin,Dinasti Qin,"Dinasti Qin (Hanzi: 秦朝, hanyu pinyin: Qin Chao..."
2,2879683,https://id.wikipedia.org/wiki/Pengusiran%20ora...,Pengusiran orang Polandia oleh Jerman Nazi,Pengusiran bangsa Polandia oleh Jerman Nazi pa...
3,949223,https://id.wikipedia.org/wiki/Politik%20Uni%20...,Politik Uni Soviet,Sistem politik Uni Soviet bercirikan kekuasaan...
4,126295,https://id.wikipedia.org/wiki/Blok%20Sekutu%20...,Blok Sekutu dalam Perang Dunia I,Blok Sekutu pada Perang Dunia I adalah negara-...



Final cleaned dataset saved to 'cleaned_historical_summarization_dataset_v2.csv'.


In [ ]:
import pandas as pd
import re
original_cleaned_df = pd.read_csv('cleaned_historical_summarization_dataset.csv')

initial_rows_count = len(original_cleaned_df)
print(f"Loaded dataset with {initial_rows_count:,} rows for refined cleaning.")

NON_CONTENT_SECTION_PATTERNS = [
    r'\b(Daftar Pustaka|Referensi|Rujukan|Pranala Luar|Lihat Pula|Sumber|Catatan Kaki|Bibliografi|Bacaan Lanjutan)\b',
    r'==\s*(Daftar Pustaka|Referensi|Rujukan|Pranala Luar|Lihat Pula|Sumber|Catatan Kaki|Bibliografi|Bacaan Lanjutan)\s*==',
    r'\b(External links|Further reading)\b' # Including some common English terms for robustness
]

combined_non_content_pattern = '|'.join(NON_CONTENT_SECTION_PATTERNS)

def truncate_non_content_sections(text, patterns):
    """Truncates the text at the first occurrence of any non-content section pattern."""
    if not isinstance(text, str): # Handle non-string types safely
        return text

    match = re.search(patterns, text, re.IGNORECASE)
    if match:
        return text[:match.start()].strip()

print("\nTruncating non-content sections...")
truncated_df = original_cleaned_df.copy()
truncated_df['text'] = truncated_df['text'].apply(lambda x: truncate_non_content_sections(x, combined_non_content_pattern))

print("Calculating word counts and filtering articles with less than 100 words...")
truncated_df['word_count'] = truncated_df['text'].apply(lambda x: len(str(x).split()))

filtered_by_word_count_df = truncated_df[truncated_df['word_count'] >= 100].reset_index(drop=True)

removed_by_truncation_count = initial_rows_count - len(truncated_df) # This should be 0 unless text became empty due to truncation.
removed_by_word_count = len(truncated_df) - len(filtered_by_word_count_df)

print("\n--- Cleaning Summary ---")
print(f"Initial number of articles: {initial_rows_count:,}")
print(f"Articles remaining after truncating non-content sections and filtering by word count: {len(filtered_by_word_count_df):,}")
print(f"Articles removed due to word count (<100 words): {removed_by_word_count:,}")

print("\n--- Sample of the final cleaned articles (first 5 rows) ---")
display(filtered_by_word_count_df[['title', 'text', 'word_count']].head())

final_output_filename = 'cleaned_historical_summarization_dataset_final.csv'
filtered_by_word_count_df.drop(columns=['word_count']).to_csv(final_output_filename, index=False)
print(f"\nFinal cleaned dataset saved to '{final_output_filename}'.")

Loaded dataset with 418 rows for refined cleaning.

Truncating non-content sections...
Calculating word counts and filtering articles with less than 100 words...

--- Cleaning Summary ---
Initial number of articles: 418
Articles remaining after truncating non-content sections and filtering by word count: 358
Articles removed due to word count (<100 words): 60

--- Sample of the final cleaned articles (first 5 rows) ---


,title,text,word_count
0,Die Tochter des Samurai,"The Daughter of the Samurai (, bahasa Jepang: ...",312
1,Bahasa Arab Mesopotamia Utara,Bahasa Arab Mesopotamia Utara (juga dikenali s...,193
2,Bahasa Akkadia,Bahasa Akkadia (lišānum akkadītum) ak-ka-du-u2...,156
3,Republik Romawi,Republik Romawi adalah sebuah negara republik ...,979
4,Sejarah Uni Soviet (1953–1964),"Di Uni Soviet, selama zaman sebelas tahun dari...",274



Final cleaned dataset saved to 'cleaned_historical_summarization_dataset_final.csv'.
